# DAX Conversational Data Validation

In [ ]:
%pip install unittest-xml-reporting

In [ ]:
workspace_id = ""
bronze_lakehouse_id = ""
databases = ""
destination_lakehouse_id = ""
destination_lakehouse_path = ""
deployment_environment = ""

In [ ]:
from pyspark.sql.functions import input_file_name, col, count, countDistinct, coalesce, lit
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, LongType
from concurrent.futures import ThreadPoolExecutor
import unittest
from pyspark.sql import SparkSession
import io
import logging
import sempy.fabric as fabric
import xmlrunner

class DaxConversationalDataIngestionTests(unittest.TestCase):

    def __init__(self, methodName='runTest', spark=None, workspace_id = None, bronze_lakehouse_id = None, databases = []):
        super().__init__(methodName)
        logging.basicConfig()
        self.logger = logging.getLogger("LOG")
        self.spark = spark
        self.workspace_id = workspace_id
        self.bronze_lakehouse_id = bronze_lakehouse_id
        self.databases = databases

    def get_transcripts_data_stats(self, root_source_data_path) -> DataFrame:
        """
        Function to get statistics on the total number of JSON records directly under all "Transcripts" subfolders,
        setting a fixed resourceType value.

        :param root_source_data_path: Root path to source data.
        :return: A DataFrame containing resourceType and total record count.
        """
        # Use recursiveFileLookup to read all JSON files under the root path
        df = self.spark.read.format("json").option("recursiveFileLookup", "true").load(root_source_data_path)

        # Filter files to include only those directly under "Transcripts" subfolders
        df = df.filter(input_file_name().rlike(r"/Transcripts/[^/]+$"))

        # Add a fixed resourceType column with the value "DaxTranscripts"
        df = df.withColumn("resourceType", lit("DaxTranscripts"))

        # Count the total number of records
        total_records = df.count()

        # Create a DataFrame with resourceType and total record count
        result_df = self.spark.createDataFrame(
            [("Transcripts", total_records)],
            schema=["resource", "record_count"]
        )

        return result_df

    def get_dax_bronze_statistics(self, delta_table_path) -> DataFrame:
        """
        Function to get statistics on the number of records per resource type within the bronze lakehouse
        
        :param delta_table_path: Path to the Delta table.
        :return: A DataFrame containing resource type, and record count
        """
        # Read the entire Delta table
        df = self.spark.read.format("delta").load(delta_table_path)
        
        # Group by resourceType and calculate counts and distinct counts
        bronze_stats_df = df.groupBy(col("parent_folder")).agg(
            count("*").alias("count")
        )
        
        return bronze_stats_df

    def get_dax_silver_statistics(self, silverlakehouse: str) -> DataFrame:
        """
        Function to get statistics on the number of records in the DaxTranscripts table within the silver lakehouse.

        :param silverlakehouse: The name of the silverlakehouse database.
        :return: A DataFrame containing the table name and record count for DaxTranscripts.
        """
        # Define schema explicitly to avoid empty schema issues
        schema = StructType([
            StructField("database", StringType(), True),
            StructField("resource", StringType(), True),
            StructField("silver_record_count", LongType(), True)
        ])
        stats_list = []
        try:
            self.spark.catalog.setCurrentDatabase(silverlakehouse)
            dax_transcripts_table = next((table for table in self.spark.catalog.listTables(silverlakehouse) if table.name == "DaxTranscripts"), None)
            if dax_transcripts_table:
                record_count = self.spark.read.table(f"{silverlakehouse}.DaxTranscripts").count()
                stats_list.append((silverlakehouse, "Transcripts", record_count))
            else:
                print(f"DaxTranscripts table not found in the silverlakehouse: {silverlakehouse}")
        except Exception as e:
            print(f"Error processing silverlakehouse {silverlakehouse}: {str(e)}")
        if not stats_list:
            return self.spark.createDataFrame([], schema)  # Return an empty DataFrame
        return self.spark.createDataFrame(stats_list, schema)       

    def compare_bronze_to_silver(self, dax_copilot_datastore_df, dax_transcripts_df) -> DataFrame:
        """
        Function to compare the count of records in the DaxCopilotDataStore table with the count of records
        in the DaxTranscripts table
        
        :param dax_copilot_datastore_df: DataFrame containing DAXCopilotDataStore table statistics.
        :param dax_transcripts_df: DataFrame containing DAXTranscripts table statistics.
        :return: A DataFrame containing resourceType, Dax bronze table record count, Dax silver table record count and the difference.
        """
        # Join the DataFrames on resourceType
        comparison_df = dax_copilot_datastore_df.join(
            dax_transcripts_df, 
            dax_copilot_datastore_df.parent_folder == dax_transcripts_df.resource, 
            "outer"
        ).select(
            coalesce(dax_copilot_datastore_df.parent_folder, dax_transcripts_df.resource).alias("resource"), 
            coalesce(col("count"), lit(0)).alias("dax_copilot_datastore_record_count"),
            coalesce(col("silver_record_count"), lit(0)).alias("dax_transcripts_record_count"),
            (coalesce(col("dax_copilot_datastore_record_count"), lit(0)) - coalesce(col("dax_transcripts_record_count"), lit(0))).alias("discrepancy_count")
        )
        
        # Check for discrepancies and print details if any
        discrepancies_df = comparison_df.filter(col("discrepancy_count") != 0)
        if discrepancies_df.count() > 0:
            print("Discrepancies found between Dax Bronze and Silver tables:")
            display(discrepancies_df)
        else:
            print("No discrepancies found between Dax Bronze and Silver tables.")

        # Order the DataFrame by record count in descending order
        comparison_df = comparison_df.orderBy(col("dax_copilot_datastore_record_count").desc(), col("dax_transcripts_record_count").desc())
        return comparison_df

    def compare_raw_source_to_bronze(self, raw_source_df, bronze_df) -> DataFrame:
        """
        Function to compare the distinct count of records in the raw source NDJSON data with the count of records
        in the bronze DaxCopilotDataStore table.
        
        :param raw_source_df: DataFrame containing raw source NDJSON statistics.
        :param bronze_df: DataFrame containing Bronze DaxCopilotDataStore statistics.
        :return: A DataFrame containing resourceType, raw source record count, bronze record count, and the difference.
        """
        # Join the DataFrames on resourceType
        comparison_df = raw_source_df.join(
            bronze_df, 
            raw_source_df.resource == bronze_df.parent_folder, 
            "outer"
        ).select(
            coalesce(raw_source_df.resource, bronze_df.parent_folder).alias("resource"),
            coalesce(col("record_count"), lit(0)).alias("raw_source_record_count"),
            coalesce(col("count"), lit(0)).alias("bronze_record_count"),
            (coalesce(col("raw_source_record_count"), lit(0)) - coalesce(col("bronze_record_count"), lit(0))).alias("discrepancy_count")
        )
        
        # Check for discrepancies and print details if any
        discrepancies_df = comparison_df.filter(col("discrepancy_count") != 0)
        if discrepancies_df.count() > 0:
            print("Discrepancies found between raw source NDJSON data and Bronze DaxCopilotDataStore table:")
            display(discrepancies_df)
        else:
            print("No discrepancies found between raw source NDJSON data and Bronze DaxCopilotDataStore table.")
        
        return comparison_df
    
    def test_raw_to_bronze_data_ingestion(self):
        
        root_source_data_path = f'abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/External/DaxCopilot/dax_copilot_data'
        source_data_stats_df = self.get_transcripts_data_stats(root_source_data_path).cache()
        bronze_delta_table_path = f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Tables/DaxCopilotData"
        bronze_statistics_df = self.get_dax_bronze_statistics(bronze_delta_table_path).cache()
        source_to_bronze_comparison_df = self.compare_raw_source_to_bronze(source_data_stats_df, bronze_statistics_df)
        
        display(source_to_bronze_comparison_df)

        # Iterate over the rows and perform the checks
        rows = source_to_bronze_comparison_df.collect()
        for row in rows:
            resource = row['resource']
            discrepancy_count = row['discrepancy_count']
            
            assert discrepancy_count == 0, f"Difference is not 0 for resource {resource}"

    def test_bronze_to_silver_data_ingestion(self):
        
        root_source_data_path = f'abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Files/External/DaxCopilot/dax_copilot_data'
        bronze_delta_table_path = f"abfss://{self.workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{self.bronze_lakehouse_id}/Tables/DaxCopilotData"
        bronze_statistics_df = self.get_dax_bronze_statistics(bronze_delta_table_path).cache()
        lakehouse_stats_df = self.get_dax_silver_statistics(databases).cache()
        comparison_df = self.compare_bronze_to_silver(bronze_statistics_df, lakehouse_stats_df)
        display(comparison_df)

        rows = comparison_df.collect()

        # Iterate over the rows and perform the checks
        for row in rows:
            resource = row['resource']
            discrepancy_count = row['discrepancy_count']
            
            assert discrepancy_count == 0, f"Difference is not 0 for resource {resource}"

def run_tests_and_write_output(spark):
    
    # Load and run tests
    loader = unittest.TestLoader()
    suite = loader.loadTestsFromTestCase(DaxConversationalDataIngestionTests)

    # Inject parameters / context to the tests
    for test in suite:
        test.spark = spark
        test.workspace_id = workspace_id
        test.bronze_lakehouse_id = bronze_lakehouse_id
        test.databases = databases

    # Write XML test report to stream, decode after completion
    write_stream = io.BytesIO()
    xmlrunner.XMLTestRunner(output=write_stream, verbosity=3).run(suite)
    xml_output = write_stream.getvalue().decode('utf-8')

    # Write report to lakehouse
    mssparkutils.fs.put(f"abfss://{workspace_id}@{deployment_environment}-onelake.dfs.fabric.microsoft.com/{destination_lakehouse_id}/{destination_lakehouse_path}", xml_output, overwrite=True)
    return xml_output

report = run_tests_and_write_output(spark)